# DeepTaxa – Tutorial 4: Architecture Deep-Dive
This notebook reproduces the [official architecture tutorial](https://systems-genomics-lab.github.io/deeptaxa/architecture.html).

It explores the `HybridCNNBERTClassifier`: its CNN and BERT branches,
fusion strategy, classification heads, and configurable variants.

In [ ]:
!pip install -q deeptaxa huggingface_hub torchinfo

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import os
os.makedirs('models', exist_ok=True)
from huggingface_hub import hf_hub_download
model_path = hf_hub_download(
    repo_id='systems-genomics-lab/deeptaxa',
    filename='deeptaxa-full-length-v1.pt',
    local_dir='models'
)
print('Model path:', model_path)

In [ ]:
# Architecture overview (ASCII diagram)
diagram = '''
Input 16S rRNA sequence (tokenised via DNABERT-2 BPE, max_length=512)
        │
        ▼
  [Embedding]  embed_dim=896
        │
   ┌────┴─────────────────────┐
   │ CNN Branch               │ BERT Branch
   │ 3 × Conv1d stacks        │ 4-layer transformer
   │ kernels=[3,5,7]          │ hidden=896, heads=7
   │ filters=256/kernel       │ intermediate=3584
   │ ReLU + Dropout(0.1)      │
   │ Global max pool          │ CLS token
   │ Linear → 896             │
   └──────────────┬───────────┘
                  │ Fusion: α·CNN + β·BERT + BERT (residual)
                  │ Dropout(0.1)
                  │
        ┌─────────┼─────────┐ ... (7 heads)
        ▼         ▼         ▼
    [Domain]  [Phylum]  [Species]
    Linear    Linear    Linear
    → n_d     → n_p     → n_s
'''
print(diagram)

In [ ]:
# Inspect checkpoint metadata
import torch
ckpt = torch.load('models/deeptaxa-full-length-v1.pt', map_location='cpu')
print('Checkpoint keys:', list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt))
if isinstance(ckpt, dict) and 'config' in ckpt:
    import pprint; pprint.pprint(ckpt['config'])

In [ ]:
# Load the model
import torch
from deeptaxa.models.hybrid import HybridCNNBERTClassifier

ckpt = torch.load('models/deeptaxa-full-length-v1.pt', map_location='cpu')
cfg  = ckpt.get('config', {})
num_labels = ckpt.get('num_labels_per_level', {})

model = HybridCNNBERTClassifier(
    tokenizer_name='zhihan1996/DNABERT-2-117M',
    num_labels_per_level=num_labels,
    **{k: v for k, v in cfg.items() if k != 'tokenizer_name'}
)
state = ckpt.get('model_state_dict', ckpt)
model.load_state_dict(state, strict=False)
model.eval()
print('Model loaded.')

In [ ]:
# Count parameters
total  = sum(p.numel() for p in model.parameters())
train  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total/1e6:.1f}M')
print(f'Trainable parameters: {train/1e6:.1f}M')

# Component breakdown
comp = {}
for name, mod in [('embedding', model.embedding),
                  ('conv_stacks', model.conv_stacks),
                  ('bert', model.bert),
                  ('classifiers', model.classifiers)]:
    comp[name] = sum(p.numel() for p in mod.parameters())/1e6
import pandas as pd
pd.DataFrame(comp.items(), columns=['Component','Params (M)'])

In [ ]:
# Model summary via torchinfo
from torchinfo import summary
B, L = 2, 128
ids  = torch.randint(0, 100, (B, L))
mask = torch.ones(B, L, dtype=torch.long)
try:
    summary(model, input_data=[ids, mask], depth=2)
except Exception as e:
    print('torchinfo summary failed:', e)
    # Fallback: manual forward
    with torch.no_grad():
        logits, _ = model(ids, mask)
    print('Output logits:', {k: v.shape for k,v in logits.items()})

In [ ]:
# Taxonomy label space per rank
if isinstance(ckpt, dict) and 'num_labels_per_level' in ckpt:
    import pandas as pd
    label_info = ckpt['num_labels_per_level']
    df = pd.DataFrame(label_info.items(), columns=['Rank','# Classes'])
    print(df.to_string(index=False))
else:
    # Reported in the paper for Greengenes2 2024.09
    print('Domain:  2  |  Phylum: 107  |  Class: 253  |  Order: 575')
    print('Family: 1063 | Genus: 3517  | Species: 9193')

In [ ]:
# Learnable fusion weights
w_cnn  = model.w_cnn.item()
w_bert = model.w_bert.item()
print(f'α (CNN weight):  {w_cnn:.4f}')
print(f'β (BERT weight): {w_bert:.4f}')
print(f'Ratio CNN/BERT:  {w_cnn/w_bert:.3f}')

In [ ]:
# Model variants comparison
import pandas as pd
variants = {
    'Variant': ['Full-length 16S','V3-V4 amplicon','V4 amplicon'],
    'Max length': [512, 300, 150],
    'Species acc (%)': [92.96, 91.4, 89.7],
    'Checkpoint': [
        'deeptaxa-full-length-v1.pt',
        'deeptaxa-v3v4-v1.pt (coming soon)',
        'deeptaxa-v4-v1.pt (coming soon)'
    ]
}
pd.DataFrame(variants)

In [ ]:
# Encoding options in DeepTaxa
print("""
DeepTaxa supports three sequence encoding strategies:

  1. dnabert  – DNABERT-2 BPE tokenizer (default, best accuracy)
               vocab_size=4096, sub-word tokenisation

  2. kmer     – k-mer frequency vectors (k=4 or k=6)
               simple, interpretable, no tokenizer needed

  3. onehot   – one-hot encoding of raw nucleotides (A/C/G/T)
               fastest, but loses positional sub-word context
""")